# Object Relationships: Association, Aggregation, and Composition

**Session:** 2026-10-30 — OOP with Python  
**Module:** 01 — Python Fundamentals & Algorithmic Thinking

In real-world software architectures, objects rarely operate in isolation. They interact through structured relationships that govern communication, coupling, and lifecycle ownership. This notebook analyzes the core object relationship taxonomy: **Association**, **Aggregation**, and **Composition**, and demonstrates how to parse and instantiate complex organizational domain models from structured JSON records.

## Learning Outcomes

- Classify relationships into Association, Aggregation, Composition, and Generalization.
- Differentiate Aggregation ("has-a", shared lifecycle) from Composition ("part-of", owner-managed lifecycle).
- Implement robust domain classes with aggregate date-of-birth attributes.
- Model multi-team corporate organizations with composite departments and aggregate employee memberships.
- Load and deserialize real-world JSON rosters (`data/organization_roster.json`) into typed object graphs.

## 1. Object Relationship Taxonomy & UML Primer

| Relationship | Semantics | Lifecycle Dependency | UML Notation | Example |
| :--- | :--- | :--- | :--- | :--- |
| **Association** | "uses-a" / communicates | Independent lifecycles | Plain solid line ($\longrightarrow$) | Mechanic services a Car |
| **Aggregation** | "has-a" / weak containment | Independent lifecycles (shared) | Clear diamond ($\diamond -$) | Team has Employees |
| **Composition** | "part-of" / strong ownership | Dependent lifecycle (cascade delete) | Filled diamond ($lacklozenge -$) | Car owns Engine; Company owns Departments |
| **Inheritance** | "is-a" / generalization | Derived type extension | Empty triangle arrow ($	riangle -$) | Cat is an Animal |

## 2. Association: Loose Peer Coupling

An **Association** represents a connection between two independent classes where neither class owns the other. Both objects have distinct lifecycles and can exist independently in the system:

In [ ]:
class Mechanic:
    """Independent service technician."""

    def __init__(self, name: str, phone: str) -> None:
        self.name = name
        self.phone = phone
        self.serviced_count: int = 0

    def service_vehicle(self, vehicle_id: str) -> str:
        self.serviced_count += 1
        return f"Mechanic {self.name} successfully inspected vehicle {vehicle_id}."


class Vehicle:
    """Vehicle operated independently of specific technicians."""

    def __init__(self, vin: str, model: str) -> None:
        self.vin = vin
        self.model = model
        self.assigned_mechanic: Mechanic | None = None

    def assign_service(self, mechanic: Mechanic) -> None:
        self.assigned_mechanic = mechanic

    def perform_maintenance(self) -> str:
        if self.assigned_mechanic is None:
            return f"Vehicle {self.vin} has no mechanic assigned."
        return self.assigned_mechanic.service_vehicle(self.vin)


# Demonstration: Both entities exist independently
tech = Mechanic("Bob Vance", "555-0199")
car = Vehicle("VIN-987654", "Electric Sedan")

car.assign_service(tech)
print(car.perform_maintenance())
assert tech.serviced_count == 1

# If the vehicle is deleted, the mechanic continues to exist
del car
print(f"Mechanic remains active: {tech.name}, total services: {tech.serviced_count}")
assert tech.serviced_count == 1

## 3. Aggregation: The "Has-A" Relationship (Shared Lifecycle)

In **Aggregation**, one object acts as a container or holder of another object, but the contained object's lifecycle is **independent** of the container:
- The aggregated object is created outside the container and passed in (via constructor or method).
- If the container object is destroyed, the aggregated object survives.
- The aggregated object can be shared concurrently across multiple containers.

Let us model a `Date` value object and a `Person` class that aggregates it:

In [ ]:
class Date:
    """Reusable calendar date value object."""

    def __init__(self, day: int, month: int, year: int) -> None:
        self.day = int(day)
        self.month = int(month)
        self.year = int(year)

    def is_same_date(self, other: "Date") -> bool:
        return self.day == other.day and self.month == other.month and self.year == other.year

    def __repr__(self) -> str:
        return f"{self.year:04d}-{self.month:02d}-{self.day:02d}"


class Person:
    """Person entity aggregating an independent Date instance."""

    def __init__(self, name: str, date_of_birth: Date) -> None:
        self.name = name
        self.date_of_birth = date_of_birth  # Aggregation: reference passed in

    def has_same_name(self, other: "Person") -> bool:
        return self.name.lower() == other.name.lower()

    def has_same_birthday(self, other: "Person") -> bool:
        return self.date_of_birth.is_same_date(other.date_of_birth)

    def describe(self) -> str:
        return f"Person(name={self.name}, DOB={self.date_of_birth})"


dob1 = Date(15, 6, 1995)
dob2 = Date(15, 6, 1995)
dob3 = Date(20, 11, 2001)

p1 = Person("Alice Smith", dob1)
p2 = Person("Bob Jones", dob2)
p3 = Person("Alice Smith", dob3)

print(p1.describe())
print(f"p1 and p2 have same birthday: {p1.has_same_birthday(p2)}")
print(f"p1 and p3 have same name:     {p1.has_same_name(p3)}")

assert p1.has_same_birthday(p2) is True
assert p1.has_same_birthday(p3) is False
assert p1.has_same_name(p3) is True

# Destroy p1: dob1 persists independently
del p1
print(f"Aggregated date object survives destruction: {dob1}")
assert dob1.year == 1995

## 4. Composition: The "Part-Of" Relationship (Strict Ownership)

In **Composition**, the composite object **owns** the lifecycle of its component parts:
- The component part is created inside the composite object (or strictly dedicated to it).
- The part cannot meaningfully exist without the composite whole.
- When the composite object is destroyed, its parts are destroyed as well (cascade deletion).

Let us model an `Employee` and their internal `CompensationProfile` (Salary):

In [ ]:
class CompensationProfile:
    """Internal salary structure tightly coupled to an employee."""

    def __init__(self, base_monthly: float, annual_bonus: float) -> None:
        self.base_monthly = float(base_monthly)
        self.annual_bonus = float(annual_bonus)

    def calculate_total_annual(self) -> float:
        return (self.base_monthly * 12.0) + self.annual_bonus


class Employee:
    """Employee owning its compensation profile via Composition."""

    def __init__(
        self,
        employee_id: str,
        name: str,
        role: str,
        email: str,
        birth_date: Date,
        base_monthly: float,
        annual_bonus: float,
    ) -> None:
        self.employee_id = employee_id
        self.name = name
        self.role = role
        self.email = email
        self.birth_date = birth_date  # Aggregation (Date can exist outside)
        # Composition: Salary profile is instantiated internally and owned exclusively
        self._compensation = CompensationProfile(base_monthly, annual_bonus)

    @property
    def annual_compensation(self) -> float:
        return self._compensation.calculate_total_annual()

    def describe(self) -> str:
        return (
            f"Employee {self.employee_id} ({self.name}), Role: {self.role}, "
            f"Annual Comp: ${self.annual_compensation:,.2f}"
        )


emp = Employee(
    employee_id="E-001",
    name="Ada Lovelace",
    role="Algorithm Architect",
    email="ada@nexus.ai",
    birth_date=Date(10, 12, 1815),
    base_monthly=12000.0,
    annual_bonus=20000.0,
)

print(emp.describe())
assert emp.annual_compensation == (12000.0 * 12 + 20000.0)

## 5. End-to-End Enterprise Case Study: Loading `organization_roster.json`

Let us load `data/organization_roster.json` to model a corporate hierarchy combining:
1. **Composition:** `Company` owns `Department` objects; `Department` owns `Team` objects.
2. **Aggregation:** `Team` aggregates `Employee` references; an employee can belong to multiple teams.

In [ ]:
import json
from pathlib import Path


class Team:
    """Team aggregating employees."""

    def __init__(self, team_id: str, name: str, mission: str) -> None:
        self.team_id = team_id
        self.name = name
        self.mission = mission
        self.members: list[Employee] = []

    def add_member(self, employee: Employee) -> None:
        if employee not in self.members:
            self.members.append(employee)

    def total_team_payroll(self) -> float:
        return sum(m.annual_compensation for m in self.members)


class Department:
    """Department composing teams."""

    def __init__(self, name: str) -> None:
        self.name = name
        self.teams: list[Team] = []

    def add_team(self, team: Team) -> None:
        self.teams.append(team)


class Company:
    """Root corporate entity owning departments."""

    def __init__(self, name: str, headquarters: str, founded_year: int) -> None:
        self.name = name
        self.headquarters = headquarters
        self.founded_year = founded_year
        self.departments: list[Department] = []

    def add_department(self, dept: Department) -> None:
        self.departments.append(dept)


# Load dataset
candidates = [
    Path("../data/organization_roster.json"),
    Path("data/organization_roster.json"),
    Path(
        "python-to-ai/01-python-fundamentals/2026-10-30-oop-with-python/data/organization_roster.json"
    ),
    Path("01-python-fundamentals/2026-10-30-oop-with-python/data/organization_roster.json"),
]
roster_path = next((p for p in candidates if p.exists()), Path("data/organization_roster.json"))
with roster_path.open("r", encoding="utf-8") as f:
    raw_data = json.load(f)

# Instantiate corporate hierarchy
company = Company(
    name=raw_data["company_name"],
    headquarters=raw_data["headquarters"],
    founded_year=raw_data["founded_year"],
)

employee_pool = {}

for dept_data in raw_data["departments"]:
    dept = Department(name=dept_data["department_name"])
    for team_data in dept_data["teams"]:
        team = Team(
            team_id=team_data["team_id"],
            name=team_data["team_name"],
            mission=team_data["mission"],
        )
        for mem in team_data["members"]:
            emp_id = mem["employee_id"]
            if emp_id not in employee_pool:
                # Instantiate shared employee object
                b_data = mem["birth_date"]
                dob = Date(b_data["day"], b_data["month"], b_data["year"])
                employee_pool[emp_id] = Employee(
                    employee_id=emp_id,
                    name=mem["name"],
                    role=mem["role"],
                    email=mem["email"],
                    birth_date=dob,
                    base_monthly=mem["salary"]["base_monthly"],
                    annual_bonus=mem["salary"]["annual_bonus"],
                )
            # Team aggregates employee reference
            team.add_member(employee_pool[emp_id])
        dept.add_team(team)
    company.add_department(dept)

print(f"Company: {company.name} ({company.headquarters})")
print(f"Departments: {len(company.departments)}, Unique Employees: {len(employee_pool)}")
for dept in company.departments:
    for t in dept.teams:
        print(
            f"  Team [{t.team_id}] {t.name}: {len(t.members)} members, Payroll: ${t.total_team_payroll():,.2f}"
        )

# Verify Sarah Connor is aggregated in both Core Algorithms and Computer Vision
sarah_teams = [
    t.name
    for d in company.departments
    for t in d.teams
    if any(m.employee_id == "EMP-101" for m in t.members)
]
print(f"Sarah Connor assigned teams: {sarah_teams}")
assert len(sarah_teams) == 2
assert "Core Algorithms" in sarah_teams
assert "Computer Vision" in sarah_teams

## Summary

- **Association:** Objects communicate as peers without ownership.
- **Aggregation:** Container holds references to independent objects that survive container destruction.
- **Composition:** Whole strictly owns parts; parts are created for and deleted with the owner.
- **Enterprise Design:** Use Composition for tightly coupled subsystems (e.g. layers in a neural network) and Aggregation for shared resources (e.g. datasets shared across dataloaders).